# CSI-FallNet: Dual-Modal Dilated Network with Cross-Modal Gating

Same philosophy as EdgeTCN (dilated causal convs, gated pooling, multi-task, distillation)  
but **dual-stream** with **shared backbone** + **cross-modal gating** + **velocity injection**.

- Student: **~50K params** | Teacher: **~160K params**
- Input: (B, 500, 232) raw CSI-Bench format (116 amp + 116 phase)
- Splits amp/phase → shared dilated backbone → cross-modal gate → velocity inject → gated pool → 3-head

In [ ]:
!pip install -q h5py onnx onnxruntime
import os,json,random,time,math,warnings,numpy as np,h5py
from pathlib import Path; from collections import Counter
import torch,torch.nn as nn,torch.nn.functional as F
from torch.utils.data import Dataset,DataLoader,WeightedRandomSampler
warnings.filterwarnings('ignore')
SEED=42; random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if torch.cuda.is_available(): torch.backends.cudnn.benchmark=True; print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'PyTorch {torch.__version__}')

In [ ]:
# ── Config ──
WIN=500; FEAT=232; N_SUB=116; T=WIN; OUT=Path('/kaggle/working')
FALL_DIR=Path('/kaggle/input/datasets/guozhenjennzhu/csi-bench/FallDetection')
if not FALL_DIR.exists():
    for p in sorted(Path('/kaggle/input').rglob('FallDetection')):
        if p.is_dir(): FALL_DIR=p; break
assert FALL_DIR.exists(), f'Not found: {FALL_DIR}'
SPLIT_DIR=FALL_DIR/'splits'
print(f'Data: {FALL_DIR}  Splits: {SPLIT_DIR.exists()}')

In [ ]:
# == Data loading (from working EdgeTCN notebook) ==
def read_csi(p):
    with h5py.File(str(p),'r') as f:
        ks=list(f.keys())
        if 'CSI_amps' in ks and 'CSI_phases' in ks:
            a=np.array(f['CSI_amps'],dtype=np.float32).squeeze()
            p_=np.array(f['CSI_phases'],dtype=np.float32).squeeze()
            if a.ndim!=2 or p_.ndim!=2: return None
            if a.shape[0]<a.shape[1]: a=a.T
            if p_.shape[0]<p_.shape[1]: p_=p_.T
            d=np.concatenate([a,p_],1)
        elif 'CSI_amps' in ks:
            d=np.array(f['CSI_amps'],dtype=np.float32).squeeze()
            if d.ndim!=2: return None
            if d.shape[0]<d.shape[1]: d=d.T
        else:
            d=np.array(f[ks[0]],dtype=np.float32).squeeze()
            if d.ndim!=2: return None
            if d.shape[0]<d.shape[1]: d=d.T
    d=np.nan_to_num(d,nan=0.,posinf=0.,neginf=0.)
    C=d.shape[1]
    if C>FEAT: d=d[:,:FEAT]
    elif C<FEAT: d=np.pad(d,((0,0),(0,FEAT-C)))
    L=d.shape[0]
    if L==WIN: return d
    elif L>WIN: s=(L-WIN)//2; return d[s:s+WIN]
    else:
        ot=np.linspace(0,1,L); nt=np.linspace(0,1,WIN)
        return np.stack([np.interp(nt,ot,d[:,k]) for k in range(FEAT)],1).astype(np.float32)

def act_label(p):
    for x in Path(str(p)).parts:
        xl=x.lower().replace('-','_')
        if xl.startswith('act_') and xl in ('act_fall','act_falling','act_fallen','act_fall_down'): return 1
    return 0

# Discover H5 files (same glob as working notebook)
files=sorted(FALL_DIR.glob('sub_Human/**/*.h5'))
if not files: files=sorted(FALL_DIR.rglob('*.h5'))
falls=[p for p in files if act_label(p)==1]; nonfalls=[p for p in files if act_label(p)==0]
print(f'H5: {len(files)} (Fall:{len(falls)} Non-Fall:{len(nonfalls)})')

# Try official splits by stem + multiple matching strategies
SPLIT_DIR=FALL_DIR/'splits'; splits={}
if SPLIT_DIR.exists():
    stem2p={p.stem:p for p in files}
    rel2p={}
    for p in files:
        try:
            r=str(p.relative_to(FALL_DIR)).replace('\\','/')
            rel2p[r]=p; rel2p[r.rsplit('.',1)[0]]=p
        except: pass
    for name in ['train','val','test','test_easy','test_medium','test_hard']:
        for sfx in ['_id.json','.json']:
            sp=SPLIT_DIR/f'{name}{sfx}'
            if not sp.exists(): continue
            ids=json.load(open(sp))
            if isinstance(ids,dict): ids=list(ids.values())[0] if ids else []
            if not isinstance(ids,list): continue
            matched=[]
            for sid in ids:
                sid=str(sid).replace('\\','/')
                fp=rel2p.get(sid) or stem2p.get(sid) or stem2p.get(Path(sid).stem)
                if fp is None:
                    for rp,rfp in rel2p.items():
                        if rp.endswith(sid) or sid.endswith(rp): fp=rfp; break
                if fp: matched.append(fp)
            if matched: splits[name]=matched; print(f'  {name:15s}: {len(matched)}')
            break
    if not splits or all(len(v)==0 for v in splits.values()):
        # Debug output
        for sfx in ['_id.json','.json']:
            sp=SPLIT_DIR/f'train{sfx}'
            if sp.exists():
                ids=json.load(open(sp))
                if isinstance(ids,dict): ids=list(ids.values())[0] if ids else []
                print(f'Split IDs sample: {ids[:3]}')
                break
        print(f'File stems sample: {[p.stem for p in files[:3]]}')
        try: print(f'File rels sample:  {[str(p.relative_to(FALL_DIR)).replace(chr(92),"/") for p in files[:3]]}')
        except: pass
        splits={}

# Fallback: random 80/20 split (same as working EdgeTCN notebook)
if not splits or 'train' not in splits or len(splits.get('train',[]))==0:
    print('Using random 80/20 split (same as EdgeTCN notebook)')
    rng=np.random.default_rng(SEED); f_=falls.copy(); n_=nonfalls.copy()
    rng.shuffle(f_); rng.shuffle(n_)
    cf=int(.8*len(f_)); cn=int(.8*len(n_))
    splits={'train':f_[:cf]+n_[:cn],'val':f_[cf:]+n_[cn:],'test':f_[cf:]+n_[cn:]}
for k in splits:
    if splits[k] and not isinstance(splits[k][0],tuple):
        splits[k]=[(p,act_label(p)) for p in splits[k]]
for k,v in splits.items(): print(f'  {k:15s}: {len(v)} (Fall:{sum(y for _,y in v)})')


In [ ]:
# == Dataset with auxiliary targets (motion_energy, quality) ==
class DS(Dataset):
    """CSI-Bench dataset. Computes auxiliary targets: motion_energy, quality."""
    def __init__(self, items, aug=False, stats=None, aux_stats=None):
        self.aug=aug; dl,ll=[],[]
        for i,(p,y) in enumerate(items):
            x=read_csi(p)
            if x is not None and x.shape==(WIN,FEAT): dl.append(x); ll.append(y)
            if (i+1)%500==0: print(f'  {i+1}/{len(items)}')
        self.X=np.stack(dl); self.Y=np.array(ll,dtype=np.float32)
        if stats is None: self.mu=self.X.mean((0,1),keepdims=True); self.sd=self.X.std((0,1),keepdims=True)+1e-6
        else: self.mu,self.sd=stats
        amp=self.X[:,:,:N_SUB]
        raw_motion=np.abs(np.diff(amp,axis=1)).mean((1,2))
        raw_quality=amp.std((1,2))
        if aux_stats is None:
            self.ml,self.mh=np.percentile(raw_motion,[2,98])
            self.ql,self.qh=np.percentile(raw_quality,[2,98])
        else: self.ml,self.mh,self.ql,self.qh=aux_stats
        self.motion=np.clip((raw_motion-self.ml)/(self.mh-self.ml+1e-6),0,1).astype(np.float32)
        self.quality=np.clip((raw_quality-self.ql)/(self.qh-self.ql+1e-6),0,1).astype(np.float32)
    def get_stats(self): return (self.mu,self.sd)
    def get_aux_stats(self): return (self.ml,self.mh,self.ql,self.qh)
    def __len__(self): return len(self.X)
    def __getitem__(self,i):
        x=self.X[i].copy()
        if self.aug:
            sh=np.random.randint(-50,51)
            if sh: x=np.roll(x,sh,0)
            x*=np.random.uniform(.9,1.1,(1,FEAT)).astype(np.float32)
            x+=np.random.normal(0,.01,x.shape).astype(np.float32)
        x=(x-self.mu.squeeze())/self.sd.squeeze()
        y=np.array([self.Y[i],self.motion[i],self.quality[i]],dtype=np.float32)
        return torch.from_numpy(x).float(),torch.from_numpy(y).float()

print('Loading train...'); tr_ds=DS(splits['train'],aug=True)
ts=tr_ds.get_stats(); aus=tr_ds.get_aux_stats()
print('Loading val...'); va_ds=DS(splits['val'],stats=ts,aux_stats=aus)
te_ds={}
for n in ['test','test_easy','test_medium','test_hard']:
    if n in splits and splits[n]: print(f'Loading {n}...'); te_ds[n]=DS(splits[n],stats=ts,aux_stats=aus)

tc=Counter(tr_ds.Y.astype(int).tolist()); print(f'Train:{len(tr_ds)} Val:{len(va_ds)} Dist:{dict(tc)}')
pos=max(1.,float(tc.get(0,1)))/max(1.,float(tc.get(1,1)))

cw={c:len(tr_ds)/(2*n) for c,n in tc.items()}
sampler=WeightedRandomSampler([cw[int(y)] for y in tr_ds.Y],len(tr_ds),True)
tr=DataLoader(tr_ds,128,sampler=sampler,num_workers=2,pin_memory=True,drop_last=True)
va=DataLoader(va_ds,256,num_workers=2,pin_memory=True)
te_loaders={n:DataLoader(d,256,num_workers=2,pin_memory=True) for n,d in te_ds.items()}


In [ ]:
# ══════════════════════════════════════════════════════════════
# Architecture: Dual-Modal Dilated Network with Cross-Modal Gating
# ══════════════════════════════════════════════════════════════
#
# NOVEL vs EdgeTCN:
#   1. Splits input into amp (116) + phase (116), not flat 232
#   2. Shared dilated backbone processes both modalities
#      -> learns modality-agnostic temporal features
#      -> halves backbone params vs separate streams
#   3. Cross-modal gating: amp & phase gate each other
#      -> learns which modality is more reliable per sample
#   4. Velocity injection: explicit temporal derivatives
#      -> gives the model acceleration info (key for falls)
#   5. 8 dilation levels (1..128) for RF=511 covering full 500 steps
#      (EdgeTCN had 4 levels for T=32)
#
# SAME as EdgeTCN:
#   - Depthwise separable dilated causal convolutions
#   - Gated pooling (attn + max + last, each sigmoid-gated)
#   - Multi-task 3-head (fall, motion_energy, quality)
#   - Teacher-student distillation
#   - Very few parameters
#

class Block(nn.Module):
    """Depthwise-separable dilated causal conv with residual.
    Causal padding ensures the output at time t only depends on t and earlier.
    Dilation d gives receptive field growth of 2*d per block."""
    def __init__(self,c,d):
        super().__init__(); self.pad=2*d
        self.dw=nn.Conv1d(c,c,3,dilation=d,groups=c,bias=False)  # depthwise: c*3 params
        self.pw=nn.Conv1d(c,c,1)                                  # pointwise: c*c params
        self.bn=nn.BatchNorm1d(c)
    def forward(self,x):
        r=x; x=F.pad(x,(self.pad,0))  # left-pad for causal
        return r+F.gelu(self.bn(self.pw(self.dw(x))))


class CSIFallNet(nn.Module):
    """Dual-modal dilated causal network.
    
    Architecture:
      amp(116) --[amp_proj]--> (c, 500) --+
                                          |--> shared dilated backbone (8 blocks)
      phase(116) -[phase_proj]-> (c, 500) -+
                                          |
      amp_feat, phase_feat                |
          |                               |
          +-- cross-modal gating ---------+
          |   ga = sigmoid(linear(amp_ctx))|
          |   gp = sigmoid(linear(pha_ctx))|
          |   fused = ga*amp + gp*phase    |
          |                               |
          +-- velocity injection ----------+
          |   vel = diff(fused, t=1)       |
          |   x = mix([fused, vel])        |
          |                               |
          +-- gated pooling (EdgeTCN) ----+
          |   a=attn_pool, m=max, l=last   |
          |   f=cat([a,m,l])               |
          |   g=cat([gate(a),gate(m),...])  |
          |   out = head(f * g)            |
          +-------------------------------+
    
    Args:
        n_sub: subcarriers per modality (116)
        c:     channel width (48=student, 96=teacher)
        drop:  dropout rate
    """
    def __init__(self, n_sub=116, c=48, drop=.08):
        super().__init__()
        self.n_sub=n_sub
        
        # Modality-specific projections (amp and phase have different distributions)
        self.amp_proj   = nn.Conv1d(n_sub, c, 1)
        self.phase_proj = nn.Conv1d(n_sub, c, 1)
        
        # Shared dilated backbone: 8 levels, RF = 2*(1+2+4+8+16+32+64+128)+1 = 511 > 500
        self.backbone = nn.Sequential(*[Block(c,d) for d in (1,2,4,8,16,32,64,128)])
        
        # Cross-modal gating: each modality's global context gates the other
        self.gate_a = nn.Linear(c, c)  # amp context -> amp gate
        self.gate_p = nn.Linear(c, c)  # phase context -> phase gate
        
        # Velocity injection: explicit temporal derivative
        self.vel_pw = nn.Conv1d(c, c, 1, bias=False)  # project velocity
        self.fuse   = nn.Sequential(nn.Conv1d(c*2, c, 1, bias=False), nn.BatchNorm1d(c))
        
        # Gated temporal pooling (identical to EdgeTCN)
        self.score = nn.Conv1d(c, 1, 1)
        h = max(6, c//8)
        self.gate = nn.Sequential(nn.Linear(c,h), nn.GELU(), nn.Linear(h,c), nn.Sigmoid())
        
        # Multi-task head: [fall_logit, motion_energy, quality]
        self.head = nn.Sequential(
            nn.LayerNorm(c*3),
            nn.Linear(c*3, 32),
            nn.GELU(),
            nn.Dropout(drop),
            nn.Linear(32, 3)
        )
    
    def forward(self, x):
        # Split into amp and phase modalities
        amp   = x[:, :, :self.n_sub].transpose(1,2)   # (B, 116, 500)
        phase = x[:, :, self.n_sub:].transpose(1,2)   # (B, 116, 500)
        
        # Project each modality into shared feature space, then encode
        af = self.backbone(self.amp_proj(amp))         # (B, c, 500)
        pf = self.backbone(self.phase_proj(phase))     # (B, c, 500)
        
        # Cross-modal gating: global context decides modality importance
        ga = torch.sigmoid(self.gate_a(af.mean(-1))).unsqueeze(-1)   # (B, c, 1)
        gp = torch.sigmoid(self.gate_p(pf.mean(-1))).unsqueeze(-1)   # (B, c, 1)
        x = ga * af + gp * pf                                        # (B, c, 500)
        
        # Velocity injection: first-order temporal derivative
        vel = F.pad(x[:,:,1:] - x[:,:,:-1], (1,0))                   # (B, c, 500)
        x = F.gelu(self.fuse(torch.cat([x, self.vel_pw(vel)], 1)))    # (B, c, 500)
        
        # Gated pooling (EdgeTCN style)
        a = (x * torch.softmax(self.score(x), -1)).sum(-1)  # attention pool
        m = x.amax(-1)                                       # max pool
        l = x[:, :, -1]                                      # last frame
        f = torch.cat([a, m, l], 1)
        g = torch.cat([self.gate(a), self.gate(m), self.gate(l)], 1)
        return self.head(f * g)                              # (B, 3)


teacher = CSIFallNet(N_SUB, 96, .10)
student = CSIFallNet(N_SUB, 48, .08)
distilled = CSIFallNet(N_SUB, 48, .08)
print(f'teacher:  {sum(p.numel() for p in teacher.parameters()):,} params')
print(f'student:  {sum(p.numel() for p in student.parameters()):,} params')
# Verify
with torch.no_grad():
    d=torch.randn(2,WIN,FEAT); o=student(d)
    print(f'input: {d.shape} -> output: {o.shape}')

In [ ]:
# ── Scoring + Training (same pattern as EdgeTCN notebook) ──

def collect(model):
    model.eval(); ys=[]; ps=[]
    with torch.no_grad():
        for x,y in va: ys.append(y[:,0].cpu()); ps.append(torch.sigmoid(model(x.to(device))[:,0]).cpu())
    return torch.cat(ys).numpy(), torch.cat(ps).numpy()

def score(model):
    """Sweep threshold for best F1 on fall class."""
    y,p=collect(model); best={'f1':-1}
    for t in np.linspace(.1,.9,81):
        z=p>=t; tp=((z)&(y==1)).sum(); fp=((z)&(y==0)).sum(); fn=((~z)&(y==1)).sum()
        pr=tp/max(1,tp+fp); re=tp/max(1,tp+fn); f=2*pr*re/max(1e-9,pr+re)
        if f>best['f1']: best={'precision':float(pr),'recall':float(re),'f1':float(f),'threshold':float(t)}
    return best

def train(model, epochs, teacher_model=None, patience=10, tag=''):
    model.to(device)
    bce=nn.BCEWithLogitsLoss(pos_weight=torch.tensor(pos,device=device))
    bce_aux=nn.BCEWithLogitsLoss()  # no pos_weight for auxiliary tasks
    opt=torch.optim.AdamW(model.parameters(), 6e-4, weight_decay=3e-4)
    best=-1.; stale=0; best_state=None
    
    for ep in range(epochs):
        model.train(); total=0.
        for x,y in tr:
            x,y=x.to(device),y.to(device)
            z=model(x)
            # Multi-task: fall (primary) + motion_energy + quality (auxiliary)
            loss = bce(z[:,0],y[:,0]) + .1*bce_aux(z[:,1],y[:,1]) + .1*bce_aux(z[:,2],y[:,2])
            # Distillation
            if teacher_model is not None:
                with torch.no_grad(): tz=teacher_model(x)
                temp=2.
                loss += .25*F.kl_div(
                    F.log_softmax(z/temp,1), F.softmax(tz/temp,1), reduction='batchmean'
                )*temp*temp
            opt.zero_grad(set_to_none=True); loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(),1.); opt.step()
            total+=loss.item()
        m=score(model)
        marker=''
        if m['f1']>best:
            best=m['f1']; best_m=m.copy()
            best_state={k:v.detach().cpu().clone() for k,v in model.state_dict().items()}
            stale=0; marker=' *'
        else: stale+=1
        print(f'{tag} ep={ep+1:02d} loss={total/max(1,len(tr)):.4f} prec={m["precision"]:.3f} rec={m["recall"]:.3f} f1={m["f1"]:.3f} thr={m["threshold"]:.2f}{marker}')
        if stale>=patience: print(f'  early stop'); break
    model.load_state_dict(best_state); return best_m

# ── Train: teacher -> student -> distilled student ──
print('='*80)
print('TRAIN TEACHER (c=96)');
teacher_m = train(teacher, 25, tag='[teacher]')
teacher.eval()
print('\nTRAIN SUPERVISED STUDENT (c=48)');
student_m = train(student, 35, tag='[student]')
print('\nTRAIN DISTILLED STUDENT (c=48)');
distilled_m = train(distilled, 35, teacher_model=teacher, tag='[distill]')

results = {'teacher': teacher_m, 'student_supervised': student_m, 'student_distilled': distilled_m}
print('\n'+json.dumps(results, indent=2))
chosen = distilled if distilled_m['f1'] >= student_m['f1'] else student
chosen_m = distilled_m if distilled_m['f1'] >= student_m['f1'] else student_m
print(f'\nChosen: {"distilled" if chosen is distilled else "supervised"} | F1={chosen_m["f1"]:.4f}')

In [ ]:
# ── Test evaluation on all splits ──

print('\n'+'='*80)
print('TEST EVALUATION')
print('='*80)

def eval_split(model, loader):
    model.eval(); all_p,all_y=[],[]
    with torch.no_grad():
        for x,y in loader:
            p=torch.sigmoid(model(x.to(device))[:,0]).cpu().numpy()
            all_p.extend(p); all_y.extend(y[:,0].numpy())
    p=np.array(all_p); y=np.array(all_y); t=chosen_m['threshold']
    z=p>=t; tp=((z)&(y==1)).sum(); fp=((z)&(y==0)).sum(); fn=((~z)&(y==1)).sum(); tn=((~z)&(y==0)).sum()
    pr=tp/max(1,tp+fp); re=tp/max(1,tp+fn); f1=2*pr*re/max(1e-9,pr+re); acc=(tp+tn)/len(y)
    # Weighted F1 (CSI-Bench metric)
    f1_0 = 2*tn/max(1,2*tn+fp+fn) if (tn+fp+fn)>0 else 0
    n0=(y==0).sum(); n1=(y==1).sum(); wf1=(f1_0*n0+f1*n1)/(n0+n1)
    return {'accuracy':float(acc),'precision':float(pr),'recall':float(re),'fall_f1':float(f1),'weighted_f1':float(wf1)}

baselines = {
    'MLP':{'f1':92.17},'ViT':{'f1':93.59},'TimesFormer1D':{'f1':93.87},
    'PatchTST':{'f1':94.03},'Transformer':{'f1':94.26},
    'ResNet-18':{'f1':94.89},'LSTM':{'f1':94.92}
}

all_test={}
for sn,ldr in te_loaders.items():
    m=eval_split(chosen,ldr); all_test[sn]=m
    print(f'\n--- {sn.upper()} ---')
    print(f'  acc={m["accuracy"]*100:.2f}%  wF1={m["weighted_f1"]*100:.2f}%  fall_F1={m["fall_f1"]*100:.2f}%  prec={m["precision"]*100:.2f}%  rec={m["recall"]*100:.2f}%')

n_params=sum(p.numel() for p in chosen.parameters())
if 'test' in all_test:
    print(f'\n{"="*60}')
    print(f'{"Model":<20s} {"wF1%":>8s} {"Params":>10s}')
    print('-'*40)
    for n,b in baselines.items(): print(f'{n:<20s} {b["f1"]:>8.2f} {"~1-11M":>10s}')
    print('-'*40)
    print(f'{"CSI-FallNet (ours)":<20s} {all_test["test"]["weighted_f1"]*100:>8.2f} {n_params:>10,}')

In [ ]:
# ── Save + ONNX export ──
torch.save({
    'model_state_dict':chosen.state_dict(),'architecture':'CSIFallNet','n_sub':N_SUB,
    'input_features':FEAT,'window_frames':WIN,'num_params':n_params,
    'metrics':chosen_m,'candidates':results,'test_results':all_test,
    'normalization':{'mean':tr_ds.mu.tolist(),'std':tr_ds.sd.tolist()},
    'threshold':chosen_m['threshold']
}, OUT/'csi_fallnet_best.pt')
print(f'Saved: {OUT/"csi_fallnet_best.pt"}')

cpu_m=chosen.to('cpu').eval()
try:
    torch.onnx.export(cpu_m,torch.zeros(1,WIN,FEAT),str(OUT/'csi_fallnet_best.onnx'),
        input_names=['csi'],output_names=['logits'],
        dynamic_axes={'csi':{0:'batch'},'logits':{0:'batch'}},opset_version=17)
    import onnxruntime as ort
    sess=ort.InferenceSession(str(OUT/'csi_fallnet_best.onnx'),providers=['CPUExecutionProvider'])
    s=np.random.randn(1,WIN,FEAT).astype(np.float32)
    pt=cpu_m(torch.from_numpy(s)).detach().numpy()
    ox=sess.run(['logits'],{'csi':s})[0]
    print(f'ONNX max|diff|={np.max(np.abs(pt-ox)):.2e}')
    times=[]
    for _ in range(100): t0=time.perf_counter(); sess.run(['logits'],{'csi':s}); times.append((time.perf_counter()-t0)*1000)
    print(f'CPU latency: median={np.median(times):.2f}ms p95={np.percentile(times,95):.2f}ms')
except Exception as e: print(f'ONNX failed: {e}')

# Summary
summary={
    'architecture':'CSI-FallNet (Dual-Modal Dilated + Cross-Modal Gating + Distillation)',
    'params':n_params,'input':[WIN,FEAT],'n_sub':N_SUB,
    'novel':['dual-stream amp/phase','shared dilated backbone','cross-modal gating','velocity injection','multi-task 3-head','teacher-student distillation'],
    'candidates':results,'test':all_test,'baselines':baselines
}
json.dump(summary,open(OUT/'csi_fallnet_results.json','w'),indent=2)
print('\n'+json.dumps(summary,indent=2))

In [ ]:
# ============================================================
# Visualization
# ============================================================
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from sklearn.metrics import confusion_matrix, roc_curve, auc

def plot_results(eval_model, loader, is_bce, threshold=0.5):
    eval_model.eval()
    all_p, all_y = [], []
    with torch.no_grad():
        for x, y in loader:
            out = eval_model(x.to(device))
            if is_bce:
                p = torch.sigmoid(out[:, 0]).cpu().numpy()
                y_true = y[:, 0].cpu().numpy() if y.ndim > 1 else y.cpu().numpy()
            else:
                p = torch.softmax(out, dim=1)[:, 1].cpu().numpy()
                y_true = y.cpu().numpy()
            all_p.extend(p)
            all_y.extend(y_true)
            
    all_p, all_y = np.array(all_p), np.array(all_y)
    preds = (all_p >= threshold).astype(int)
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
    
    # Confusion Matrix
    cm = confusion_matrix(all_y, preds)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax1, 
                xticklabels=['Non-Fall', 'Fall'], yticklabels=['Non-Fall', 'Fall'])
    ax1.set_title('Test Split Confusion Matrix')
    ax1.set_ylabel('True Label')
    ax1.set_xlabel('Predicted Label')
    
    # ROC Curve
    fpr, tpr, _ = roc_curve(all_y, all_p)
    ax2.plot(fpr, tpr, color='darkorange', lw=2, label=f'AUC = {auc(fpr, tpr):.3f}')
    ax2.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
    ax2.set_xlabel('False Positive Rate'); ax2.set_ylabel('True Positive Rate')
    ax2.set_title('ROC Curve')
    ax2.legend(loc="lower right")
    
    plt.tight_layout()
    plt.show()

# Auto-detect parameters based on notebook
target_model = chosen if 'chosen' in locals() else model
bce_mode = 'chosen' in locals()  # FallNet uses 'chosen' and BCE
thr = chosen_m['threshold'] if 'chosen_m' in locals() else 0.5

if 'te_loaders' in locals() and 'test' in te_loaders:
    eval_loader = te_loaders['test']
elif 'test_loaders' in locals() and 'test' in test_loaders:
    eval_loader = test_loaders['test']
else:
    eval_loader = val_loader

print("Generating visual evaluation...")
plot_results(target_model, eval_loader, bce_mode, thr)
